# AlphaCluster — Optuna Hyperparameter Tuning

Automated Bayesian optimization of reward scales, curriculum multipliers, and PPO hyperparameters.

**Strategy:** Two-phase optimization:
1. **Phase 1 — Screening:** 200k steps × 40 trials (TPE + MedianPruner)
2. **Phase 2 — Validation:** 500k steps × top 10 configs

**Parameters:** 13 dimensions (8 reward scales + 2 curriculum multipliers + 3 PPO)

All results persist to Google Drive with resume support.

In [ ]:
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/alphacluster")
DRIVE_SRC = DRIVE_ROOT / "src"

assert DRIVE_SRC.exists(), (
    f"Source not found at {DRIVE_SRC}\n"
    f"Copy your local src/ directory to Google Drive: My Drive/alphacluster/src/"
)

LOCAL_SRC = Path("/content/src")
if LOCAL_SRC.exists():
    shutil.rmtree(LOCAL_SRC)
shutil.copytree(DRIVE_SRC, LOCAL_SRC)
print(f"Copied source to {LOCAL_SRC}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "stable-baselines3>=2.4,<3.0", "gymnasium>=1.0,<2.0",
    "pyarrow", "python-dotenv", "tqdm", "rich",
    "optuna>=3.0", "plotly", "kaleido"], check=True)

sys.path.insert(0, str(LOCAL_SRC))

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

DRIVE_DIR = Path("/content/drive/MyDrive/AlphaCluster/optuna_tuning/")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB = DRIVE_DIR / "optuna_study.db"
RESULTS_CSV = DRIVE_DIR / "trial_results.csv"
BEST_PARAMS_JSON = DRIVE_DIR / "best_params.json"
PLOTS_DIR = DRIVE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPersistence: {DRIVE_DIR}")

## Cell 2 — Load & Split Data

In [ ]:
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/alphacluster/data")
KLINES_PATH = DATA_DIR / "btcusdt_5m.parquet"
FUNDING_PATH = DATA_DIR / "btcusdt_funding.parquet"

assert KLINES_PATH.exists(), f"Kline data not found at {KLINES_PATH}"

klines_df = pd.read_parquet(KLINES_PATH, engine="pyarrow")
print(f"Loaded {len(klines_df):,} candles")

funding_df = None
if FUNDING_PATH.exists():
    funding_df = pd.read_parquet(FUNDING_PATH, engine="pyarrow")
    print(f"Loaded {len(funding_df):,} funding records")

n = len(klines_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = klines_df.iloc[:train_end].reset_index(drop=True)
val_df = klines_df.iloc[train_end:val_end].reset_index(drop=True)

print(f"Data split: train={len(train_df):,}  val={len(val_df):,}")

## Cell 3 — Define Objective Function

In [ ]:
import logging
import shutil
import numpy as np
import optuna

from alphacluster.agent.config import TrainingConfig
from alphacluster.agent.trainer import (
    CurriculumCallback,
    TrainingMetricsCallback,
    create_agent,
)
from alphacluster.env.trading_env import TradingEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

logger = logging.getLogger("optuna_tuning")

SCREENING_TIMESTEPS = 200_000
VALIDATION_TIMESTEPS = 500_000
METRICS_LOG_FREQ = 100_000
N_ENVS = 4


class OptunaPruningCallback(BaseCallback):
    """Report intermediate scores to Optuna and prune bad trials."""

    def __init__(self, trial, metrics_path, log_freq):
        super().__init__(verbose=0)
        self.trial = trial
        self.metrics_path = metrics_path
        self.log_freq = log_freq
        self._next_report = log_freq

    def _on_step(self):
        if self.num_timesteps < self._next_report:
            return True
        self._next_report += self.log_freq
        if self.metrics_path.exists():
            score, _ = compute_score(self.metrics_path)
            self.trial.report(score, self.num_timesteps)
            if self.trial.should_prune():
                raise optuna.TrialPruned()
        return True


def make_env(df, funding_df, config, rank=0):
    """Factory for SubprocVecEnv."""
    def _init():
        import os, warnings
        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
        env = TradingEnv(
            df=df, funding_df=funding_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
        )
        env.reset(seed=rank)
        return env
    return _init


def compute_score(metrics_path):
    """Read last row of training metrics CSV and compute composite score."""
    df = pd.read_csv(metrics_path)
    if df.empty:
        return -1000.0, {}

    row = df.iloc[-1]
    flat_pct = row.get("flat_pct", 100.0)
    trades = row.get("trades_per_episode", 0.0)
    duration = row.get("avg_trade_duration", 0.0)
    total_pnl = row.get("total_pnl_pct", 0.0)
    win_rate = row.get("win_rate", 0.0)

    if flat_pct > 70:
        return -1000.0, {"reject": "flat_pct > 70%"}
    if trades < 20:
        return -1000.0, {"reject": f"trades={trades} < 20"}
    if duration < 1.5:
        return -1000.0, {"reject": f"duration={duration} < 1.5"}

    pnl_norm = np.clip(total_pnl, -50, 50) / 100 + 0.5
    trades_norm = min(trades, 200) / 200
    win_rate_norm = win_rate / 100

    score = pnl_norm * 0.4 + trades_norm * 0.3 + win_rate_norm * 0.3

    details = {
        "total_pnl_pct": total_pnl, "trades_per_episode": trades,
        "win_rate": win_rate, "flat_pct": flat_pct,
        "avg_trade_duration": duration,
        "pnl_norm": pnl_norm, "trades_norm": trades_norm,
        "win_rate_norm": win_rate_norm,
    }
    return float(score), details


def objective(trial):
    """Optuna objective: sample params, train, evaluate."""
    train_env = None
    try:
        fee_scale = trial.suggest_float("fee_scale", 0.1, 2.0)
        opp_cost_scale = trial.suggest_float("opportunity_cost_scale", 0.1, 2.0)
        opp_cost_cap = trial.suggest_float("opportunity_cost_cap", 0.01, 0.15)
        opp_cost_threshold = trial.suggest_float(
            "opportunity_cost_threshold", 0.001, 0.005,
        )
        churn_scale = trial.suggest_float("churn_penalty_scale", 0.1, 2.0)
        dd_scale = trial.suggest_float("drawdown_penalty_scale", 0.1, 2.0)
        quality_scale = trial.suggest_float("quality_scale", 0.1, 2.0)
        pos_mgmt_scale = trial.suggest_float("position_mgmt_scale", 0.1, 2.0)
        p3_fee_mult = trial.suggest_float("phase3_fee_multiplier", 1.0, 3.0)
        p3_churn_mult = trial.suggest_float("phase3_churn_multiplier", 1.0, 3.0)
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
        ent = trial.suggest_float("ent_coef_phase1", 0.01, 0.1, log=True)
        gamma = trial.suggest_float("gamma", 0.99, 0.999)

        base_reward_config = {
            "fee_scale": fee_scale,
            "opportunity_cost_scale": opp_cost_scale,
            "opportunity_cost_cap": opp_cost_cap,
            "opportunity_cost_threshold": opp_cost_threshold,
            "churn_penalty_scale": churn_scale,
            "drawdown_penalty_scale": dd_scale,
            "quality_scale": quality_scale,
            "position_mgmt_scale": pos_mgmt_scale,
        }

        config = TrainingConfig(
            total_timesteps=SCREENING_TIMESTEPS,
            learning_rate=lr,
            gamma=gamma,
            ent_coef=ent,
            eval_freq=SCREENING_TIMESTEPS + 1,
        )

        envs = SubprocVecEnv([
            make_env(train_df, funding_df, config, rank=i)
            for i in range(N_ENVS)
        ])
        train_env = VecNormalize(envs, norm_obs=True, norm_reward=False)

        eval_raw = TradingEnv(
            df=val_df, funding_df=funding_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
        )

        agent = create_agent(train_env, config, verbose=0)

        trial_dir = Path(f"/content/optuna_trial_{trial.number}")
        trial_dir.mkdir(parents=True, exist_ok=True)
        metrics_path = trial_dir / "training_metrics.csv"

        metrics_cb = TrainingMetricsCallback(
            eval_env=eval_raw,
            log_freq=METRICS_LOG_FREQ,
            n_episodes=3,
            log_path=metrics_path,
            verbose=0,
        )
        curriculum_cb = CurriculumCallback(
            config,
            base_reward_config=base_reward_config,
            phase3_fee_multiplier=p3_fee_mult,
            phase3_churn_multiplier=p3_churn_mult,
            ent_coef_phase1=ent,
            verbose=0,
        )
        pruning_cb = OptunaPruningCallback(trial, metrics_path, METRICS_LOG_FREQ)

        agent.learn(
            total_timesteps=config.total_timesteps,
            callback=CallbackList([metrics_cb, curriculum_cb, pruning_cb]),
            progress_bar=False,
        )

        score, details = compute_score(metrics_path)
        for k, v in details.items():
            trial.set_user_attr(k, v)

        shutil.rmtree(trial_dir, ignore_errors=True)
        return score

    except optuna.TrialPruned:
        raise
    except Exception as e:
        trial.set_user_attr("error", str(e))
        logger.exception("Trial %d failed: %s", trial.number, e)
        return -1000.0
    finally:
        if train_env is not None:
            try:
                train_env.close()
            except Exception:
                pass

print("Objective function defined. Ready for Phase 1.")

## Cell 4 — Phase 1: Screening (200k × 40 trials)

In [ ]:
print("=" * 60)
print("PHASE 1: Screening — 200k steps × 40 trials")
print("=" * 60)

study = optuna.create_study(
    study_name="alphacluster_reward_tuning",
    direction="maximize",
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=100_000,
    ),
    sampler=optuna.samplers.TPESampler(seed=42),
)

completed = len([t for t in study.trials
                 if t.state in (optuna.trial.TrialState.COMPLETE,
                                optuna.trial.TrialState.PRUNED)])
remaining = max(0, 40 - completed)
print(f"Completed trials: {completed}, remaining: {remaining}")

if remaining > 0:
    study.optimize(objective, n_trials=remaining, timeout=None)
else:
    print("All 40 screening trials already completed.")

print(f"\nBest trial: #{study.best_trial.number}, score={study.best_trial.value:.4f}")

## Cell 5 — Phase 1 Results & Analysis

In [ ]:
import json

results = []
for trial in study.trials:
    if trial.value is not None and trial.value > -999:
        row = {"trial": trial.number, "score": trial.value, "state": trial.state.name}
        row.update(trial.params)
        row.update(trial.user_attrs)
        results.append(row)

results_df = pd.DataFrame(results).sort_values("score", ascending=False)
print(f"Viable trials: {len(results_df)} / {len(study.trials)}")
print()
print(results_df.head(10).to_string(index=False))

results_df.to_csv(RESULTS_CSV, index=False)
print(f"\nResults saved to {RESULTS_CSV}")

try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
        plot_slice,
    )

    fig1 = plot_optimization_history(study)
    fig1.write_image(str(PLOTS_DIR / "phase1_optimization_history.png"))
    fig1.show()

    fig2 = plot_param_importances(study)
    fig2.write_image(str(PLOTS_DIR / "phase1_param_importances.png"))
    fig2.show()

    fig3 = plot_slice(study)
    fig3.write_image(str(PLOTS_DIR / "phase1_slice.png"))
    fig3.show()

    print(f"Plots saved to {PLOTS_DIR}")
except Exception as e:
    print(f"Visualization error: {e}")

## Cell 6 — Phase 2: Validation (500k × top 10)

In [ ]:
print("=" * 60)
print("PHASE 2: Validation — 500k steps × top 10")
print("=" * 60)

top_trials = sorted(
    [t for t in study.trials if t.value is not None and t.value > -999],
    key=lambda t: t.value,
    reverse=True,
)[:10]

print(f"Validating {len(top_trials)} configs at {VALIDATION_TIMESTEPS:,} timesteps\n")

VALIDATION_CSV = DRIVE_DIR / "validation_results.csv"
validation_results = []

existing_validated = set()
if VALIDATION_CSV.exists():
    existing_df = pd.read_csv(VALIDATION_CSV)
    existing_validated = set(existing_df["screening_trial"].tolist())
    validation_results = existing_df.to_dict("records")

for i, trial_data in enumerate(top_trials):
    if trial_data.number in existing_validated:
        print(f"  [{i+1}/10] Trial #{trial_data.number} — already validated, skipping")
        continue

    print(f"  [{i+1}/10] Trial #{trial_data.number} (screening score={trial_data.value:.4f})")
    params = trial_data.params

    train_env = None
    try:
        base_reward_config = {
            "fee_scale": params["fee_scale"],
            "opportunity_cost_scale": params["opportunity_cost_scale"],
            "opportunity_cost_cap": params["opportunity_cost_cap"],
            "opportunity_cost_threshold": params["opportunity_cost_threshold"],
            "churn_penalty_scale": params["churn_penalty_scale"],
            "drawdown_penalty_scale": params["drawdown_penalty_scale"],
            "quality_scale": params["quality_scale"],
            "position_mgmt_scale": params["position_mgmt_scale"],
        }

        config = TrainingConfig(
            total_timesteps=VALIDATION_TIMESTEPS,
            learning_rate=params["learning_rate"],
            gamma=params["gamma"],
            ent_coef=params["ent_coef_phase1"],
            eval_freq=VALIDATION_TIMESTEPS + 1,
        )

        envs = SubprocVecEnv([
            make_env(train_df, funding_df, config, rank=r)
            for r in range(N_ENVS)
        ])
        train_env = VecNormalize(envs, norm_obs=True, norm_reward=False)

        eval_raw = TradingEnv(
            df=val_df, funding_df=funding_df,
            window_size=config.window_size,
            episode_length=config.episode_length,
        )

        agent = create_agent(train_env, config, verbose=0)

        trial_dir = Path(f"/content/validation_trial_{trial_data.number}")
        trial_dir.mkdir(parents=True, exist_ok=True)
        metrics_path = trial_dir / "training_metrics.csv"

        metrics_cb = TrainingMetricsCallback(
            eval_env=eval_raw, log_freq=METRICS_LOG_FREQ,
            n_episodes=3, log_path=metrics_path, verbose=0,
        )
        curriculum_cb = CurriculumCallback(
            config,
            base_reward_config=base_reward_config,
            phase3_fee_multiplier=params["phase3_fee_multiplier"],
            phase3_churn_multiplier=params["phase3_churn_multiplier"],
            ent_coef_phase1=params["ent_coef_phase1"],
            verbose=0,
        )

        agent.learn(
            total_timesteps=config.total_timesteps,
            callback=CallbackList([metrics_cb, curriculum_cb]),
            progress_bar=False,
        )

        score, details = compute_score(metrics_path)
        result = {
            "screening_trial": trial_data.number,
            "screening_score": trial_data.value,
            "validation_score": score,
            **details,
            **params,
        }
        validation_results.append(result)

        pd.DataFrame(validation_results).to_csv(VALIDATION_CSV, index=False)

        shutil.rmtree(trial_dir, ignore_errors=True)

        print(f"    → validation score: {score:.4f}")

    except Exception as e:
        print(f"    → FAILED: {e}")
        validation_results.append({
            "screening_trial": trial_data.number,
            "screening_score": trial_data.value,
            "validation_score": -1000.0,
            "error": str(e),
        })
        pd.DataFrame(validation_results).to_csv(VALIDATION_CSV, index=False)
    finally:
        if train_env is not None:
            try:
                train_env.close()
            except Exception:
                pass

## Cell 7 — Final Results & Export

In [ ]:
import json

val_df_results = pd.DataFrame(validation_results)
val_df_results = val_df_results.sort_values("validation_score", ascending=False)

print("=" * 60)
print("FINAL RESULTS — Ranked by 500k Validation Score")
print("=" * 60)
print()

display_cols = [
    "screening_trial", "screening_score", "validation_score",
    "total_pnl_pct", "trades_per_episode", "win_rate", "flat_pct",
]
available = [c for c in display_cols if c in val_df_results.columns]
print(val_df_results[available].to_string(index=False))

best_row = val_df_results.iloc[0]
best_trial_num = int(best_row["screening_trial"])
best_trial = next(t for t in study.trials if t.number == best_trial_num)

best_params = {
    "source": "optuna_tuning",
    "screening_trial": best_trial_num,
    "screening_score": float(best_row["screening_score"]),
    "validation_score": float(best_row["validation_score"]),
    "params": best_trial.params,
    "base_reward_config": {
        k: best_trial.params[k]
        for k in [
            "fee_scale", "opportunity_cost_scale", "opportunity_cost_cap",
            "opportunity_cost_threshold", "churn_penalty_scale",
            "drawdown_penalty_scale", "quality_scale", "position_mgmt_scale",
        ]
    },
    "curriculum": {
        "phase3_fee_multiplier": best_trial.params["phase3_fee_multiplier"],
        "phase3_churn_multiplier": best_trial.params["phase3_churn_multiplier"],
        "ent_coef_phase1": best_trial.params["ent_coef_phase1"],
    },
    "ppo": {
        "learning_rate": best_trial.params["learning_rate"],
        "gamma": best_trial.params["gamma"],
    },
}

with open(BEST_PARAMS_JSON, "w") as f:
    json.dump(best_params, f, indent=2)

print(f"\nBest params saved to {BEST_PARAMS_JSON}")
print(f"\nBest configuration (trial #{best_trial_num}):")
print(json.dumps(best_params["params"], indent=2))

## Cell 8 — Apply Best Parameters (Optional)

In [ ]:
import json

with open(BEST_PARAMS_JSON) as f:
    bp = json.load(f)

print("=" * 60)
print("HOW TO APPLY BEST PARAMETERS")
print("=" * 60)
print()
print("In your training notebook (colab_train.ipynb), modify Cell 3:")
print()
print(f"config = TrainingConfig(")
print(f"    total_timesteps=2_000_000,")
print(f"    learning_rate={bp['ppo']['learning_rate']},")
print(f"    gamma={bp['ppo']['gamma']},")
print(f"    ent_coef={bp['curriculum']['ent_coef_phase1']},")
print(f")")
print()
print(f"base_reward_config = {json.dumps(bp['base_reward_config'], indent=4)}")
print()
print("# Pass to CurriculumCallback:")
print(f"curriculum_cb = CurriculumCallback(")
print(f"    config,")
print(f"    base_reward_config=base_reward_config,")
print(f"    phase3_fee_multiplier={bp['curriculum']['phase3_fee_multiplier']},")
print(f"    phase3_churn_multiplier={bp['curriculum']['phase3_churn_multiplier']},")
print(f"    ent_coef_phase1={bp['curriculum']['ent_coef_phase1']},")
print(f")")